In [5]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [6]:
players = pd.read_csv("data/raw/players.csv")

tm = players[players["last_season"] >= 2022].copy()
print(tm.shape)

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd

In [ ]:
players = pd.read_csv("data/raw/players.csv")

tm = players[players["last_season"] >= 2022].copy()
print(tm.shape)

In [ ]:
import unicodedata, re

def normalise(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-z\s]", "", name.lower())
    return re.sub(r"\s+", " ", name).strip()

df["name_norm"] = df["player_name"].apply(normalise)
tm["name_norm"] = tm["name"].apply(normalise)

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/understat_all.csv")
df = df[df["time"] >= 900].copy()

stats = ["goals", "xG", "npxG", "assists", "xA", "shots",
         "key_passes", "xGChain", "xGBuildup"]
for col in stats:
    df[f"{col}_p90"] = df[col] / df["time"] * 90

df["pos_clean"] = df["position"].str.replace("S", "").str.strip()
df["is_mid"] = df["pos_clean"].str.contains("M")

print(df.shape)

In [ ]:
players = pd.read_csv("data/raw/players.csv")
tm = players[players["last_season"] >= 2022].copy()
print(tm.shape)

In [7]:
df[["player_name", "name_norm"]].head()

NameError: name 'df' is not defined

In [8]:
B
import pandas as pd
import unicodedata, re

# Understat data
df = pd.read_csv("data/raw/understat_all.csv")
df = df[df["time"] >= 900].copy()

stats = ["goals", "xG", "npxG", "assists", "xA", "shots",
         "key_passes", "xGChain", "xGBuildup"]
for col in stats:
    df[f"{col}_p90"] = df[col] / df["time"] * 90

df["pos_clean"] = df["position"].str.replace("S", "").str.strip()
df["is_mid"] = df["pos_clean"].str.contains("M")

# Transfermarkt data
players = pd.read_csv("data/raw/players.csv")
tm = players[players["last_season"] >= 2022].copy()

# Name cleaning
def normalise(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-z\s]", "", name.lower())
    return re.sub(r"\s+", " ", name).strip()

df["name_norm"] = df["player_name"].apply(normalise)
tm["name_norm"] = tm["name"].apply(normalise)

print(df.shape, tm.shape)

NameError: name 'B' is not defined

In [9]:
import pandas as pd
import unicodedata, re

# Understat data
df = pd.read_csv("data/raw/understat_all.csv")
df = df[df["time"] >= 900].copy()

stats = ["goals", "xG", "npxG", "assists", "xA", "shots",
         "key_passes", "xGChain", "xGBuildup"]
for col in stats:
    df[f"{col}_p90"] = df[col] / df["time"] * 90

df["pos_clean"] = df["position"].str.replace("S", "").str.strip()
df["is_mid"] = df["pos_clean"].str.contains("M")

# Transfermarkt data
players = pd.read_csv("data/raw/players.csv")
tm = players[players["last_season"] >= 2022].copy()

# Name cleaning
def normalise(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-z\s]", "", name.lower())
    return re.sub(r"\s+", " ", name).strip()

df["name_norm"] = df["player_name"].apply(normalise)
tm["name_norm"] = tm["name"].apply(normalise)

print(df.shape, tm.shape)

(4799, 32) (32336, 27)


In [10]:
df[["player_name", "name_norm"]].head()

,player_name,name_norm
0,Erling Haaland,erling haaland
1,Harry Kane,harry kane
2,Ivan Toney,ivan toney
3,Mohamed Salah,mohamed salah
4,Callum Wilson,callum wilson


In [12]:
tm_unique = (tm.sort_values("market_value_in_eur", ascending=False)
               .drop_duplicates("name_norm", keep="first"))

In [13]:
lookup = tm_unique.set_index("name_norm")["player_id"].to_dict()

names = df[["player_name", "name_norm"]].drop_duplicates().copy()
names["player_id"] = names["name_norm"].map(lookup)

print(names["player_id"].notna().mean())

0.9204192546583851


In [14]:
from rapidfuzz import process, fuzz

In [15]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [16]:
tm_names = tm_unique["name_norm"].tolist()

def fuzzy(name):
    res = process.extractOne(name, tm_names, scorer=fuzz.token_sort_ratio, score_cutoff=88)
    return (res[0], res[1]) if res else (None, None)

todo = names["player_id"].isna()
results = names.loc[todo, "name_norm"].apply(fuzzy)

names["tm_name_norm"] = None
names["score"] = None
names.loc[todo, "tm_name_norm"] = results.str[0]
names.loc[todo, "score"] = results.str[1]
names.loc[todo, "player_id"] = names.loc[todo, "tm_name_norm"].map(lookup)

print(names["player_id"].notna().mean())

0.9483695652173914


In [17]:
fuzzy_matched = names[names["score"].notna() & names["player_id"].notna()].copy()
fuzzy_matched["score"] = fuzzy_matched["score"].astype(float)
fuzzy_matched[["player_name", "tm_name_norm", "score"]].sort_values("score").head(30)

,player_name,tm_name_norm,score
3304,Marco Faraoni,marco farfan,88.000000
123,Rasmus Kristensen,rasmus carstensen,88.235294
838,Illia Zabarnyi,ilya zabarnyi,88.888889
611,Oliver McBurnie,oli mcburnie,88.888889
2220,Jan-Niklas Beste,niklas beste,88.888889
2285,Gian-Luca Waldschmidt,luca waldschmidt,88.888889
6820,Nicolás Melamed,nico melamed,88.888889
5566,Mousa Al Tamari,mousa tamari,88.888889
8022,Bertug Yildirim,bertug yldrm,88.888889
3210,Nicolás González,nico gonzalez,89.655172


In [18]:
names[names["player_id"].isna()]["player_name"].head(30)

6                  Martin Odegaard
86                      Bobby Reid
154                    Naif Aguerd
167              Joe Ayodele-Aribo
202                          Jonny
212                      Estupiñán
235     Benoit Badiashile Mukinayi
355                           Kepa
410                     Max Kilman
422               Ezri Konsa Ngoyo
442          Cheick Oumar Doucoure
451                   Matthew Cash
683                      Pape Sarr
736         Iyenoma Destiny Udogie
814           Valentino Livramento
833                 Vinicius Souza
840                Ameen Al Dakhil
937                    Louis Beyer
1162            Amad Diallo Traore
1180              Emile Smith-Rowe
1350           Chimuanya Ugochukwu
1527          Stefan Ortega Moreno
1751                     Bote Baku
1756             Silas Wamangituka
1772             Jordan Siebatcheu
1778      Leandro Barreiro Martins
1800       Eric Junior Dina Ebimbe
1825                  Santos Borré
1899                

In [19]:
from rapidfuzz import process, fuzz

def normalise(name):
    for a, b in {"ø":"o","Ø":"O","æ":"ae","Æ":"AE","ß":"ss","đ":"d","Đ":"D","ł":"l","Ł":"L"}.items():
        name = str(name).replace(a, b)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.lower().replace("-", " ")
    name = re.sub(r"[^a-z\s]", "", name)
    return re.sub(r"\s+", " ", name).strip()

df["name_norm"] = df["player_name"].apply(normalise)
tm["name_norm"] = tm["name"].apply(normalise)

tm_unique = (tm.sort_values("market_value_in_eur", ascending=False)
               .drop_duplicates("name_norm", keep="first"))
lookup = tm_unique.set_index("name_norm")["player_id"].to_dict()
tm_names = tm_unique["name_norm"].tolist()

names = df[["player_name", "name_norm"]].drop_duplicates().copy()
names["player_id"] = names["name_norm"].map(lookup)
names["method"] = names["player_id"].notna().map({True: "exact", False: None})
names["tm_name_norm"] = None
names["score"] = None

def run_pass(scorer, cutoff, label, min_words=1):
    todo = names["player_id"].isna()
    for idx in names[todo].index:
        n = names.at[idx, "name_norm"]
        if len(n.split()) < min_words:
            continue
        res = process.extractOne(n, tm_names, scorer=scorer, score_cutoff=cutoff)
        if res:
            names.at[idx, "tm_name_norm"] = res[0]
            names.at[idx, "score"] = res[1]
            names.at[idx, "player_id"] = lookup[res[0]]
            names.at[idx, "method"] = label

run_pass(fuzz.token_sort_ratio, 88, "fuzzy1")
run_pass(fuzz.token_set_ratio, 95, "fuzzy2", min_words=2)

print(names["player_id"].notna().mean())
print(names["method"].value_counts())

0.9798136645962733
method
exact     2390
fuzzy2      77
fuzzy1      57
Name: count, dtype: int64


In [20]:
names[names["method"].isin(["fuzzy1", "fuzzy2"])][["player_name", "tm_name_norm", "score", "method"]].sort_values("score").head(40)

,player_name,tm_name_norm,score,method
3304,Marco Faraoni,marco farfan,88.0,fuzzy1
123,Rasmus Kristensen,rasmus carstensen,88.235294,fuzzy1
5566,Mousa Al Tamari,mousa tamari,88.888889,fuzzy1
8022,Bertug Yildirim,bertug yldrm,88.888889,fuzzy1
611,Oliver McBurnie,oli mcburnie,88.888889,fuzzy1
6820,Nicolás Melamed,nico melamed,88.888889,fuzzy1
838,Illia Zabarnyi,ilya zabarnyi,88.888889,fuzzy1
3520,Berat Gjimshiti,berat djimsiti,89.655172,fuzzy1
3210,Nicolás González,nico gonzalez,89.655172,fuzzy1
6689,Santiago Comesaña,santi comesana,90.322581,fuzzy1


In [21]:
tm[tm["name"].str.contains("Arrizabalaga")][["player_id", "name", "last_season", "market_value_in_eur"]]

,player_id,name,last_season,market_value_in_eur
12889,192279,Kepa Arrizabalaga,2025,5000000.0


In [22]:
manual = {"kepa": 12345}   # example only, use the real player_id
for n, pid in manual.items():
    names.loc[names["name_norm"] == n, "player_id"] = pid

In [23]:
import pandas as pd
import unicodedata, re

In [24]:
print(names["method"].value_counts(dropna=False))

method
exact     2390
fuzzy2      77
fuzzy1      57
NaN         52
Name: count, dtype: int64


In [25]:
check = names[names["method"].isin(["fuzzy1", "fuzzy2"])]
check[["player_name", "tm_name_norm", "score", "method"]].sort_values("method").to_string()

'                      player_name             tm_name_norm      score  method\n17                  Son Heung-Min            heung min son      100.0  fuzzy1\n5602                  Lee Kang-In              kang in lee      100.0  fuzzy1\n5566              Mousa Al Tamari             mousa tamari  88.888889  fuzzy1\n5560                 Yusuf Yazici               yusuf yazc  90.909091  fuzzy1\n5276               Dmitri Lienard          dimitri lienard  96.551724  fuzzy1\n5260             Andrew Gravillon        andreaw gravillon  96.969697  fuzzy1\n5154            Azz-Eddine Ounahi          azzedine ounahi      93.75  fuzzy1\n5151              Hichem Boudaoui          hicham boudaoui  93.333333  fuzzy1\n5743               Darline Yongwa            darlin yongwa  96.296296  fuzzy1\n5030                       Fabián                  fabiano  92.307692  fuzzy1\n4453           Benjamín Domínguez          benja dominguez  90.909091  fuzzy1\n4405               Kephren Thuram          khephren

In [26]:
check[["player_name", "tm_name_norm", "score", "method"]].head(70)

,player_name,tm_name_norm,score,method
17,Son Heung-Min,heung min son,100.0,fuzzy1
86,Bobby Reid,bobby de cordova reid,100.0,fuzzy2
123,Rasmus Kristensen,rasmus carstensen,88.235294,fuzzy1
167,Joe Ayodele-Aribo,joe aribo,100.0,fuzzy2
235,Benoit Badiashile Mukinayi,benoit badiashile,100.0,fuzzy2
...,...,...,...,...
4968,Arnaud Kalimuendo Muinga,arnaud kalimuendo,100.0,fuzzy2
4970,Muhammed-Cham Saracevic,muhammed cham,100.0,fuzzy2
4978,Mohamed Youcef Belaïli,youcef belaili,100.0,fuzzy2
4992,Joia Nuno Da Costa,nuno da costa,100.0,fuzzy2


In [27]:
check[["player_name", "tm_name_norm", "score", "method"]].tail(64)

,player_name,tm_name_norm,score,method
5025,Ben Seghir,eliesse ben seghir,100.0,fuzzy2
5030,Fabián,fabiano,92.307692,fuzzy1
5052,Alexsandro Ribeiro,alexsandro,100.0,fuzzy2
5080,Pereira Lage,mathias pereira lage,100.0,fuzzy2
5151,Hichem Boudaoui,hicham boudaoui,93.333333,fuzzy1
...,...,...,...,...
7998,Marcos de Sousa,de,100.0,fuzzy2
8022,Bertug Yildirim,bertug yldrm,88.888889,fuzzy1
8050,Adrián,adriano,92.307692,fuzzy1
8183,Urko Gonzalez,urko gonzalez de zarate,100.0,fuzzy2


In [28]:
names[names["player_id"].isna()][["player_name"]].head(52)

,player_name
154,Naif Aguerd
202,Jonny
212,Estupiñán
451,Matthew Cash
814,Valentino Livramento
937,Louis Beyer
1350,Chimuanya Ugochukwu
1751,Bote Baku
1899,Kouadio Kone
2025,Dayotchanculle Upamecano


In [29]:
bad = ["name one", "name two"]   # use the name_norm values of the wrong ones
names.loc[names["name_norm"].isin(bad), ["player_id", "method"]] = None

In [30]:
fz = names["method"].isin(["fuzzy1", "fuzzy2"])
tm_words = names["tm_name_norm"].fillna("").str.split().str.len()
bad = fz & (tm_words < 2)

print(names[bad][["player_name", "tm_name_norm"]].to_string())

names.loc[bad, ["player_id", "method", "tm_name_norm", "score"]] = None
print(names["method"].value_counts(dropna=False))

                player_name tm_name_norm
410              Max Kilman          max
833          Vinicius Souza        souza
1756      Silas Wamangituka        silas
1772      Jordan Siebatcheu       jordan
1825           Santos Borré       santos
2211          Álex Grimaldo         alex
2838            Jean Négoce         jean
3541            Kelvin Adou       kelvin
4878        Kialonda Gaspar       gaspar
5030                 Fabián      fabiano
5052     Alexsandro Ribeiro   alexsandro
6317               Mat Ryan         ryan
6612         Ezequiel Ávila     ezequiel
6795         Anthony Lozano      anthony
6837             Álex Balde         alex
7036         Isaac Carcelén        isaac
7195         Samu Omorodion         samu
7815  Juan Camilo Hernández         juan
7907          Anuar Mohamed        anuar
7998        Marcos de Sousa           de
8050                 Adrián      adriano
method
exact     2390
NaN         73
fuzzy2      58
fuzzy1      55
Name: count, dtype: int64


In [31]:
pd.set_option("display.max_rows", 200)

check = names[names["method"].isin(["fuzzy1", "fuzzy2"])]
check[["player_name", "tm_name_norm", "score", "method"]].sort_values("score")

,player_name,tm_name_norm,score,method
3304,Marco Faraoni,marco farfan,88.0,fuzzy1
123,Rasmus Kristensen,rasmus carstensen,88.235294,fuzzy1
611,Oliver McBurnie,oli mcburnie,88.888889,fuzzy1
8022,Bertug Yildirim,bertug yldrm,88.888889,fuzzy1
5566,Mousa Al Tamari,mousa tamari,88.888889,fuzzy1
6820,Nicolás Melamed,nico melamed,88.888889,fuzzy1
838,Illia Zabarnyi,ilya zabarnyi,88.888889,fuzzy1
3210,Nicolás González,nico gonzalez,89.655172,fuzzy1
3520,Berat Gjimshiti,berat djimsiti,89.655172,fuzzy1
6956,Alejandro Pozo,alejandro pozuelo,90.322581,fuzzy1


In [32]:
wrong = ["juan herzog"]   # name_norm values of wrong pairs
names.loc[names["name_norm"].isin(wrong), ["player_id", "method", "tm_name_norm", "score"]] = None

In [33]:
ids = names.drop_duplicates("name_norm")[["name_norm", "player_id"]]
df = df.drop(columns=["player_id"], errors="ignore").merge(ids, on="name_norm", how="left")

print(df["player_id"].notna().mean())

0.9724942696395082


In [34]:
df = df[df["player_id"].notna()].copy()
df["player_id"] = df["player_id"].astype(int)
print(df.shape)

(4667, 33)


In [35]:
dob = players[["player_id", "date_of_birth"]].copy()
dob["date_of_birth"] = pd.to_datetime(dob["date_of_birth"], errors="coerce")

df = df.merge(dob, on="player_id", how="left")
df["season_end"] = pd.to_datetime((df["season"].astype(int) + 1).astype(str) + "-06-30")
df["age"] = ((df["season_end"] - df["date_of_birth"]).dt.days / 365.25).round(1)

df[["player_name", "season", "age"]].head()

,player_name,season,age
0,Erling Haaland,2022,22.9
1,Harry Kane,2022,29.9
2,Ivan Toney,2022,27.3
3,Mohamed Salah,2022,31.0
4,Callum Wilson,2022,31.3


In [36]:
values = pd.read_csv("data/raw/player_valuations.csv")
values["date"] = pd.to_datetime(values["date"])
values = values[["player_id", "date", "market_value_in_eur"]].sort_values("date")

df = df.sort_values("season_end")
df = pd.merge_asof(
    df, values,
    left_on="season_end", right_on="date",
    by="player_id", direction="backward",
    tolerance=pd.Timedelta("365D")
)
df = df.rename(columns={"market_value_in_eur": "market_value"})

In [37]:
print(df[["age", "market_value"]].isna().mean())

df[df["player_name"] == "Martin Odegaard"][["player_name", "season", "age", "market_value"]]

age             0.000857
market_value    0.016713
dtype: float64


,player_name,season,age,market_value
1082,Martin Odegaard,2022,24.5,90000000.0
3105,Martin Odegaard,2023,25.5,110000000.0
3331,Martin Odegaard,2024,26.5,85000000.0


In [38]:
import os
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/players_merged.csv", index=False)
print(df.shape)

(4667, 38)


In [39]:
df[["age", "market_value"]].describe()

,age,market_value
count,4663.000000,4.589000e+03
mean,27.328866,1.577206e+07
std,4.084151,2.050897e+07
min,15.200000,5.000000e+04
25%,24.300000,3.500000e+06
50%,27.000000,8.000000e+06
75%,30.200000,2.000000e+07
max,41.700000,2.000000e+08


In [40]:
df.duplicated(["player_id", "season"]).sum()

np.int64(32)

In [41]:
import os
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/players_merged.csv", index=False)
print(df.shape)

(4667, 38)


In [42]:
df[["age", "market_value"]].describe()

,age,market_value
count,4663.000000,4.589000e+03
mean,27.328866,1.577206e+07
std,4.084151,2.050897e+07
min,15.200000,5.000000e+04
25%,24.300000,3.500000e+06
50%,27.000000,8.000000e+06
75%,30.200000,2.000000e+07
max,41.700000,2.000000e+08


In [43]:
df.duplicated(["player_id", "season"]).sum()

np.int64(32)

In [44]:
dups = df[df.duplicated(["player_id", "season"], keep=False)]
dups[["player_name", "season", "team_title", "league", "time"]].sort_values(["player_name", "season"])

,player_name,season,team_title,league,time
3905,Anthony Rouault,2024,VfB Stuttgart,Bundesliga,1223
4425,Anthony Rouault,2024,Rennes,Ligue_1,979
3599,Antonio Candela,2024,Real Valladolid,La_Liga,935
3842,Antonio Candela,2024,Venezia,Serie_A,1020
207,Benjamin Lecomte,2022,Montpellier,Ligue_1,1710
1506,Benjamin Lecomte,2022,Espanyol,La_Liga,900
613,Dango Ouattara,2022,Lorient,Ligue_1,1560
1123,Dango Ouattara,2022,Bournemouth,EPL,1254
893,Danilo,2022,Juventus,Serie_A,3174
1235,Danilo,2022,Nottingham Forest,EPL,999


In [45]:
count_cols = ["games", "time", "goals", "xG", "assists", "xA", "shots",
              "key_passes", "yellow_cards", "red_cards", "npg", "npxG",
              "xGChain", "xGBuildup"]
keep_cols = ["player_name", "team_title", "league", "position", "pos_clean",
             "is_mid", "age", "market_value", "date_of_birth", "season_end", "name_norm"]

df = df.sort_values("time", ascending=False)

agg = {c: "sum" for c in count_cols}
agg.update({c: "first" for c in keep_cols})

df = df.groupby(["player_id", "season"], as_index=False).agg(agg)

In [46]:
stats = ["goals", "xG", "npxG", "assists", "xA", "shots",
         "key_passes", "xGChain", "xGBuildup"]
for col in stats:
    df[f"{col}_p90"] = df[col] / df["time"] * 90

print(df.duplicated(["player_id", "season"]).sum())
print(df.shape)

0
(4635, 36)


In [47]:
df.to_csv("data/processed/players_merged.csv", index=False)

In [48]:
git add .
git commit -m "Combine mid-season transfers into one row per player-season"
git push

SyntaxError: invalid syntax (2281453950.py, line 1)

In [49]:
git add .
git commit -m "Combine mid-season transfers into one row per player-season"
git push

SyntaxError: invalid syntax (2281453950.py, line 1)

In [50]:
print(df.duplicated(["player_id", "season"]).sum())
print(df.shape)

0
(4635, 36)


In [51]:
cd ~/Projects/football-scouting
git add .
git commit -m "Combine mid-season transfers into one row per player-season"
git push

SyntaxError: invalid syntax (1725637611.py, line 1)

In [52]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
